# Solvers ⚙️

In this exercise, we will explore the effects of different `solvers` on `LogisticRegression` models.

👇 Run the code below to import the dataset

In [1]:
import pandas as pd

df = pd.read_csv("https://d32aokrjazspmn.cloudfront.net/materials/solvers_dataset.csv")
df.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,sulphates,alcohol,quality rating
0,9.47,5.97,7.36,10.17,6.84,9.15,9.78,9.52,10.34,8.80,6
1,10.05,8.84,9.76,8.38,10.15,6.91,9.70,9.01,9.23,8.80,7
2,10.59,10.71,10.84,10.97,9.03,10.42,11.46,11.25,11.34,9.06,4
3,11.00,8.44,8.32,9.65,7.87,10.92,6.97,11.07,10.66,8.89,8
4,12.12,13.44,10.35,9.95,11.09,9.38,10.22,9.04,7.68,11.38,3


- The dataset consists of different wines 🍷
- The features describe different qualities of the wines
- The target 🎯 is a quality rating given by an expert

## 1. Target Engineering

In this section, we will convert the ratings into a binary target.

👇 How many observations are there for each rating?

In [2]:
df["quality rating"].unique()

array([ 6,  7,  4,  8,  3,  1,  2, 10,  5,  9])

In [4]:
df["quality rating"].value_counts()

quality rating
10    10143
5     10124
1     10090
2     10030
8      9977
6      9961
9      9955
7      9954
4      9928
3      9838
Name: count, dtype: int64

❓ Create `y` by converting the target into a binary classification task, where quality ratings below 6 are bad [0] and ratings of 6 and above are good [1]

In [7]:
y = df['quality rating'].map(lambda x: 0 if x < 6 else 1)

❓ Check the class balance of the new binary target

In [8]:
y.value_counts()

quality rating
0    50010
1    49990
Name: count, dtype: int64

❓ Create your `X` by normalizing the features. This will allow for a fair comparison of different solvers.

In [9]:
X = df.drop(columns=['quality rating'])

In [10]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler().fit(X)

X_scaled = scaler.transform(X)

X_scaled = pd.DataFrame(X_scaled)
X_scaled.columns = X.columns

## 2. LogisticRegression Solvers

❓ Logistic Regression models can be optimized using different **solvers**. Compare the available solvers:
- Fit time - which solver is the **fastest**?
- Accuracy - how **different** are the accuracy scores?

Available solvers for Logistic Regression: `['newton-cg', 'lbfgs', 'liblinear', 'sag', 'saga']`
 
For more information about these 5 solvers, check out [this Stack Overflow thread](https://stackoverflow.com/questions/38640109/logistic-regression-python-solvers-defintions)

In [13]:
from sklearn.model_selection import cross_validate
from sklearn.linear_model import LogisticRegression

solvers = ['newton-cg', 'lbfgs', 'liblinear', 'sag', 'saga']

scores = []
fit_times = []

for solver in solvers:
    cv_results_sv = cross_validate(
        LogisticRegression(solver=solver), 
        X_scaled, y, 
        cv=5, scoring=["precision"]
        )

    scores.append(cv_results_sv['test_precision'].mean())
    fit_times.append(cv_results_sv['fit_time'].mean())

solvers_performance = pd.DataFrame({"precision score":scores, "fit time": fit_times}, index = solvers)
solvers_performance

,precision score,fit time
newton-cg,0.874383,0.085530
lbfgs,0.874200,0.053936
liblinear,0.874449,0.150397
sag,0.874378,0.316839
saga,0.874386,0.498499


In [17]:
# YOUR ANSWER
fastest_solver = solvers_performance["fit time"].idxmin()
fastest_solver

'lbfgs'

<details>
    <summary>ℹ️ Click here for our take</summary>

Since our cost function is "easy" enough to have a global minimum that all 5 solvers can find, all solvers should produce similar accuracy scores. For very complex cost functions like in Deep Learning, different solvers may stop at different values of the loss function.

**Wine dataset**
    
If you check the feature importance with sklearn's <a href="https://scikit-learn.org/stable/modules/generated/sklearn.inspection.permutation_importance.html">permutation_importance</a> on the current dataset, you will see that many features have nearly 0 importance. The liblinear solver moves in only *one* direction at a time and regularizes the others with L1 regularization (i.e., sets beta values to 0), which can provide a good fit for a dataset where many features are not that important for predicting the target.

❗️There is a cost to searching for the best solver. Sticking with the default (`lbfgs`) can generally save the most time; sklearn offers this table to give you an idea of which solver to choose to get started: 

<img src="https://wagon-public-datasets.s3.amazonaws.com/05-Machine-Learning/04-Under-the-Hood/solvers-chart.png" width=700>

</details>

### 🧪 Test your code

In [18]:
from nbresult import ChallengeResult

result = ChallengeResult(
    'solvers',
    fastest_solver=fastest_solver
)
result.write()
print(result.check())


============================= test session starts ==============================
platform darwin -- Python 3.12.9, pytest-8.3.4, pluggy-1.5.0 -- /Users/yaren/.pyenv/versions/workintech/bin/python
cachedir: .pytest_cache
rootdir: /Users/yaren/code/ds_projects/data-solvers/tests
plugins: anyio-4.12.1, dash-4.0.0, typeguard-4.4.2
collecting ... collected 1 item

test_solvers.py::TestSolvers::test_fastest_solver PASSED                 [100%]

============================== 1 passed in 0.01s ===============================


💯 You can commit your code:

git add tests/solvers.pickle

git commit -m 'Completed solvers step'

git push origin master



## 3. Stochastic Gradient Descent

Logistic Regression models can also be optimized with Stochastic Gradient Descent.

❓ Evaluate a Logistic Regression model optimized with **Stochastic Gradient Descent**. How do the accuracy score and training time compare to the performance of the models trained in section 2?

<details>
<summary>💡 Hint</summary>

- If you get stuck, check the [SGDClassifier documentation](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.SGDClassifier.html)!

</details>

In [20]:
from sklearn.linear_model import SGDClassifier

sgd_model = SGDClassifier(loss="log_loss")

cv_results_sgd = cross_validate(sgd_model, X_scaled, y, cv=5, scoring=["precision"])

In [24]:
cv_results_sgd["test_precision"].mean()

0.8707839668578176

In [23]:
cv_results_sgd["fit_time"].mean()

0.1567638874053955

☝️ The SGD model should have one of the shortest fit times for similar performance (it can even be shorter than `liblinear`). This is a direct effect of performing each epoch of Gradient Descent on a single row at a time instead of loading 100k rows into memory at once.

## 4. Predictions

❓ Using the best model (balanced between short fit time and high accuracy), predict the binary quality (0 or 1) of the wine below. Save the following:
- `predicted_class`
- `predicted_proba_of_class` (i.e., if your model predicted class 1, what is the probability that class 1 is correct, should be between 0 and 1)

In [25]:
new_wine = pd.read_csv('https://d32aokrjazspmn.cloudfront.net/materials/solvers_new_wine.csv')
new_wine

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,sulphates,alcohol
0,9.54,13.5,12.35,8.78,14.72,9.06,9.67,10.15,11.17,12.17


In [26]:
best_model = SGDClassifier(loss="log_loss").fit(X_scaled,y)

new_X = pd.DataFrame(scaler.transform(new_wine))
new_X.columns = new_wine.columns

best_model.predict(new_X)

array([0])

In [28]:
predicted_class = best_model.predict(new_X)
predicted_class

array([0])

In [30]:
predicted_proba_of_class = best_model.predict_proba(new_X)[0, predicted_class]
predicted_proba_of_class

array([0.96567527])

# 🏁  Check your code and submit your notebook

In [31]:
from nbresult import ChallengeResult

result = ChallengeResult(
    'new_data_prediction',
    predicted_class=predicted_class,
    predicted_proba_of_class=predicted_proba_of_class
)
result.write()
print(result.check())


============================= test session starts ==============================
platform darwin -- Python 3.12.9, pytest-8.3.4, pluggy-1.5.0 -- /Users/yaren/.pyenv/versions/workintech/bin/python
cachedir: .pytest_cache
rootdir: /Users/yaren/code/ds_projects/data-solvers/tests
plugins: anyio-4.12.1, dash-4.0.0, typeguard-4.4.2
collecting ... collected 2 items

test_new_data_prediction.py::TestNewDataPrediction::test_predicted_class PASSED [ 50%]
test_new_data_prediction.py::TestNewDataPrediction::test_predicted_proba PASSED [100%]

============================== 2 passed in 0.14s ===============================


💯 You can commit your code:

git add tests/new_data_prediction.pickle

git commit -m 'Completed new_data_prediction step'

git push origin master

